<a href="https://colab.research.google.com/github/GGSimmons1992/UTYV6k8pXAuLL0yL/blob/main/Notebooks/retrieveMask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# retrieveMask.ipynb
## Goal 1: Extract the final feature mask chosen by the RL agent
## Goal 1 success criterion: match or beat the baseline F1 score using fewer features

In [1]:
!pip install category_encoders
!pip install stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 8.3 MB/s eta 0:00:00


In [2]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier as rf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback
from google.colab import drive

drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Src/')

import dataPrep
import classicRF
from salesReinforcerEnvironments import FeatureSelectionEnv

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
def evaluate_model(train_df, dev_df, feature_names, hyperparameters, target_column="isSubscribed"):
    if len(feature_names) == 0:
        return {
            "accuracy": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0
        }

    model = rf(
        n_estimators=hyperparameters["n_estimators"],
        max_features=hyperparameters["max_features"],
        criterion=hyperparameters["criterion"],
        max_depth=hyperparameters["max_depth"],
        random_state=42
    )

    X_train = train_df[feature_names]
    y_train = train_df[target_column]
    X_dev = dev_df[feature_names]
    y_dev = dev_df[target_column]

    model.fit(X_train, y_train)
    predictions = model.predict(X_dev)

    return {
        "accuracy": float(accuracy_score(y_dev, predictions)),
        "precision": float(precision_score(y_dev, predictions, zero_division=0)),
        "recall": float(recall_score(y_dev, predictions, zero_division=0)),
        "f1": float(f1_score(y_dev, predictions, zero_division=0))
    }

In [4]:
def rollout_final_feature_state(env, model):
    obs, _ = env.reset()
    done = False

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    return env.unwrapped

In [5]:
def extract_final_artifact(env):
    final_mask = env.feature_mask.copy()
    selected_feature_indices = np.where(final_mask == 1)[0]
    selected_features = [env.original_feature_names[i] for i in selected_feature_indices]

    final_hyperparameters = {
        "n_estimators": int(env.n_estimators),
        "max_features": env.max_features_options[env.max_features_idx],
        "criterion": env.criterion_options[env.criterion_idx],
        "max_depth": int(env.max_depth)
    }

    return {
        "feature_mask": final_mask.tolist(),
        "selected_features": selected_features,
        "hyperparameters": final_hyperparameters
    }

In [6]:
def build_baseline_hyperparameters(base_model):
    return {
        "n_estimators": int(base_model.n_estimators),
        "max_features": base_model.max_features,
        "criterion": base_model.criterion,
        "max_depth": int(base_model.max_depth) if base_model.max_depth is not None else None
    }

In [7]:
def compare_baseline_vs_rl(train_df, dev_df, baseline_hyperparameters, rl_artifact, all_feature_names):
    baseline_metrics = evaluate_model(train_df, dev_df, all_feature_names, baseline_hyperparameters)
    rl_metrics = evaluate_model(
        train_df,
        dev_df,
        rl_artifact["selected_features"],
        rl_artifact["hyperparameters"]
    )

    baseline_feature_count = len(all_feature_names)
    rl_feature_count = len(rl_artifact["selected_features"])

    metric_rows = []
    for metric_name in ["accuracy", "precision", "recall", "f1"]:
        metric_rows.append({
            "metric": metric_name,
            "baseline": baseline_metrics[metric_name],
            "rl": rl_metrics[metric_name],
            "difference": rl_metrics[metric_name] - baseline_metrics[metric_name]
        })

    comparison_df = pd.DataFrame(metric_rows)

    return {
        "baseline": {
            "metrics": baseline_metrics,
            "feature_count": baseline_feature_count,
            "hyperparameters": baseline_hyperparameters
        },
        "rl": {
            "metrics": rl_metrics,
            "feature_count": rl_feature_count,
            "feature_mask": rl_artifact["feature_mask"],
            "selected_features": rl_artifact["selected_features"],
            "hyperparameters": rl_artifact["hyperparameters"]
        },
        "goal_1_success": rl_metrics["f1"] >= baseline_metrics["f1"] and rl_feature_count < baseline_feature_count,
        "all_metrics_comparable_or_better": all(
            rl_metrics[metric_name] >= baseline_metrics[metric_name]
            for metric_name in ["accuracy", "precision", "recall", "f1"]
        ),
        "comparison_table": comparison_df
    }

In [8]:
class GoalF1EarlyStopCallback(BaseCallback):
  def __init__(
      self,
      eval_env,
      train_df,
      dev_df,
      baseline_hyperparameters,
      all_feature_names,
      target_f1=0.90,
      eval_freq=2000,
      consecutive_hits=3,
      verbose=1
  ):
      super().__init__(verbose)
      self.eval_env = eval_env
      self.train_df = train_df
      self.dev_df = dev_df
      self.baseline_hyperparameters = baseline_hyperparameters
      self.all_feature_names = all_feature_names
      self.target_f1 = target_f1
      self.eval_freq = eval_freq
      self.consecutive_hits = consecutive_hits
      self.hit_streak = 0
      self.best_f1 = 0.0
      self.final_report = None

  def _on_step(self):
      if self.num_timesteps % self.eval_freq != 0:
          return True

      final_env = rollout_final_feature_state(self.eval_env, self.model)
      rl_artifact = extract_final_artifact(final_env)
      report = compare_baseline_vs_rl(
          self.train_df,
          self.dev_df,
          self.baseline_hyperparameters,
          rl_artifact,
          self.all_feature_names
      )

      rl_f1 = report["rl"]["metrics"]["f1"]
      if rl_f1 > self.best_f1:
          self.best_f1 = rl_f1

      if rl_f1 >= self.target_f1:
          self.hit_streak += 1
      else:
          self.hit_streak = 0

      if self.verbose > 0:
          print(
              f"[Eval @ {self.num_timesteps}] RL F1={rl_f1:.4f}, "
              f"best_F1={self.best_f1:.4f}, streak={self.hit_streak}/{self.consecutive_hits}"
          )

      if self.hit_streak >= self.consecutive_hits:
          self.final_report = report
          if self.verbose > 0:
              print(
                  f"Early stop: RL F1 reached target {self.target_f1:.2f} "
                  f"for {self.consecutive_hits} consecutive evaluations."
              )
          return False

      return True

In [9]:
def main():
    full_train_data = dataPrep.retrieveCSVFromDrive("SalesReinforcerTrain.csv")
    train, dev = train_test_split(
        full_train_data,
        test_size=0.2,
        random_state=42,
        stratify=full_train_data["isSubscribed"]
    )

    feature_env = FeatureSelectionEnv(train_df=train, dev_df=dev)
    feature_env.feature_penalty_weight = 0.01
    feature_env.max_steps = 200

    baseline_model = classicRF.retrieveModelFromDrive("baseRandomForest.pkl")
    baseline_hyperparameters = build_baseline_hyperparameters(baseline_model)
    all_feature_names = feature_env.original_feature_names

    early_stop_callback = GoalF1EarlyStopCallback(
        eval_env=feature_env,
        train_df=train,
        dev_df=dev,
        baseline_hyperparameters=baseline_hyperparameters,
        all_feature_names=all_feature_names,
        target_f1=0.90,
        eval_freq=2000,
        consecutive_hits=3,
        verbose=1
    )

    model = DQN(
        "MultiInputPolicy",
        feature_env,
        learning_rate=1e-4,
        buffer_size=200_000,
        learning_starts=5_000,
        batch_size=256,
        gamma=0.995,
        train_freq=4,
        gradient_steps=1,
        target_update_interval=2000,
        exploration_fraction=0.4,
        exploration_final_eps=0.02,
        verbose=1,
        seed=42
    )

    model.learn(total_timesteps=150_000, callback=early_stop_callback)

    if early_stop_callback.final_report is not None:
        goal_1_report = early_stop_callback.final_report
    else:
        final_env = rollout_final_feature_state(feature_env, model)
        rl_artifact = extract_final_artifact(final_env)
        goal_1_report = compare_baseline_vs_rl(
            train,
            dev,
            baseline_hyperparameters,
            rl_artifact,
            all_feature_names
        )

    comparison_table = goal_1_report.pop("comparison_table")
    print(json.dumps(goal_1_report, indent=2))
    display(comparison_table)

    report_path = "/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Models/goal_1_report.json"
    comparison_table_path = "/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Models/goal_1_metrics_comparison.csv"

    with open(report_path, "w") as report_file:
        json.dump(goal_1_report, report_file, indent=2)

    comparison_table.to_csv(comparison_table_path, index=False)

    print(f"Saved Goal 1 report to {report_path}")
    print(f"Saved Goal 1 comparison table to {comparison_table_path}")

In [10]:
if __name__ == "__main__":
    main()

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 200      |
|    ep_rew_mean      | 86.4     |
|    exploration_rate | 0.987    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 2        |
|    time_elapsed     | 285      |
|    total_timesteps  | 800      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 200      |
|    ep_rew_mean      | 89.5     |
|    exploration_rate | 0.974    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 2        |
|    time_elapsed     | 550      |
|    total_timesteps  | 1600     |
----------------------------------
[Eval @ 2000] RL F1=0.0000, best_F1=0.0000, streak=0/3
----------------------------------
| rollout/            |          |
|    ep_le

TypeError: Object of type int64 is not JSON serializable